#### aim: from different airgo file format versions, we transform data to common format called 'airgo reserach'

In [1]:
import numpy as np
import pandas as pd
import re
import datetime
import os


In [2]:
# sample research airgo, as done for sleep lab data:
sample_airgo_research = pd.read_csv('C:/Users/wg984/Wolfgang/repos/ICU-Sleep/data/sample_airgo_research.csv')
sample_airgo_research = sample_airgo_research.head()
sample_airgo_research

,DateTime,Band,accX,accY,accZ,CRCStatus
0,2019-08-03 22:26:32.378,3115.0,1.15,1.75,-9.56,NaN
1,2019-08-03 22:26:32.478,3109.0,1.11,1.96,-9.51,NaN
2,2019-08-03 22:26:32.578,3109.0,1.11,1.71,-9.42,NaN
3,2019-08-03 22:26:32.678,3101.0,0.91,1.73,-9.43,NaN
4,2019-08-03 22:26:32.778,3095.0,1.01,1.77,-9.57,NaN


In [73]:
folder = '001_Sullivan'
re.split('_', folder)[0]

'001'

In [65]:
### get all v4 breath files:
savefolder = '//mad3/MGH-NEURO-CDAC/Projects/ICU_SLEEP_STUDY/data/data_analysis/AirGo/airgo_research_single_files'
savepaths = []
    
### get all v4 breath files:
enrolled = '//mad3/MGH-NEURO-CDAC/Projects/ICU_SLEEP_STUDY/data/enrolled_subjects/'
enrolled_folders = os.listdir(enrolled)
v4_files = []
for folder in enrolled_folders[:170]:
    if '.db' in folder: continue
    if '.txt' in folder: continue
    files = os.listdir(os.path.join(enrolled, folder, 'airgo'))
    #     print(files)
    for file in files:
        if '_Breath.csv' in file:
            v4_files.append(os.path.join(enrolled, folder, 'airgo', file))
            savepaths.append(os.path.join(savefolder, file))


### for v4 raw:

In [45]:
def airgo_v4_to_research(filepath, savepath):
    AirGo_data = pd.read_csv(filepath)
    
    AirGo_data.columns = AirGo_data.columns.str.strip()
    print(AirGo_data.columns)
    # delete first entry because it has weird values, e.g. breathHigh = 0. also sampleTime+breathDuration is sometimes lower than sample time of second entry.
    AirGo_data.drop(0,0, inplace=True)

    AirGo_breath = pd.DataFrame()

    sampleTime_beginning = AirGo_data['sampleTime']
    sampleTime_end = AirGo_data['sampleTime'] + AirGo_data['breathDuration']
    sampleTime_raw = pd.Series(np.zeros(2*len(sampleTime_beginning)))
    sampleTime_raw.iloc[0::2] = sampleTime_beginning.values
    sampleTime_raw.iloc[1::2] = sampleTime_end.values

    girth = pd.Series(np.zeros(len(sampleTime_raw)))
    girth.iloc[0::2] = AirGo_data['breathLow'].values
    girth.iloc[1::2] = AirGo_data['breathHigh'].values

    accX = pd.Series(np.zeros(len(sampleTime_raw)))
    accX.iloc[0::2] = AirGo_data['accX'].values
    accX.iloc[1::2] = AirGo_data['accX'].values

    accY = pd.Series(np.zeros(len(sampleTime_raw)))
    accY.iloc[0::2] = AirGo_data['accY'].values
    accY.iloc[1::2] = AirGo_data['accY'].values

    accZ = pd.Series(np.zeros(len(sampleTime_raw)))
    accZ.iloc[0::2] = AirGo_data['accZ'].values
    accZ.iloc[1::2] = AirGo_data['accZ'].values

    # sample time in seconds:
    AirGo_breath['sampleTime'] = sampleTime_raw / 1000 
    AirGo_breath['girth'] = girth
    AirGo_breath['accX'] = accX
    AirGo_breath['accY'] = accY
    AirGo_breath['accZ'] = accZ

    # create the DateTime array:

    AirGo_breath['sampleTime']

    starting_date = re.search("\d\d\d\d-\d\d-\d\d \d\d\d\d\d\d", filename)
    starting_date = starting_date.group(0)
    starting_date = datetime.datetime.strptime(starting_date, '%Y-%m-%d %H%M%S')

    DateTime = pd.Series( np.zeros(AirGo_breath.shape[0],) )
    DateTime.index = AirGo_breath.index

    for counter, sampleTime in enumerate(AirGo_breath['sampleTime']):

        DateTime.iloc[counter] = starting_date + datetime.timedelta(seconds=sampleTime)

    AirGo_breath['DateTime'] = DateTime

    AirGo_breath.columns = ['sampleTime','Band','accX','accY','accZ','DateTime']
    AirGo_breath = AirGo_breath[['DateTime','Band','accX','accY','accZ']]

    # resampling:
    AirGo_breath = AirGo_breath.set_index('DateTime');
    AirGo_breath.index = AirGo_breath.index.round('100ms')
    resampled = AirGo_breath.resample(datetime.timedelta(seconds=0.1)).interpolate(method='pchip', order =3, limit_direction='forward', axis=0)
    resampled.reset_index(level=0, inplace=True)
    resampled = resampled[['DateTime','Band','accX','accY','accZ']]
    
    # drop duplicate:
    dup = resampled.duplicated(subset=['DateTime','Band'])
    resampled = resampled.mask(dup).dropna(axis = 0, how='all')
	
	# detect all open belt areas
	OpenBeltIdx = np.where(resampled['Band'] > 60000)[0]

	LastIdxOpenBeltAtBeginOfRecording = np.where(np.diff(OpenBeltIdx)>1)[0][0]
	LastIdxOpenBeltAtBeginOfRecording = OpenBeltIdx[LastIdxOpenBeltAtBeginOfRecording]
	FirstIdxOpenBeltAtEndOfRecording = np.where(np.diff(OpenBeltIdx)>1)[0][-1]
	FirstIdxOpenBeltAtEndOfRecording  = OpenBeltIdx[FirstIdxOpenBeltAtEndOfRecording+1]
	resampled = resampled.iloc[LastIdxOpenBeltAtBeginOfRecording+1:FirstIdxOpenBeltAtEndOfRecording,:]


    resampled['CRCStatus'] = np.nan
    
    resampled.to_csv(savepath, index = False)
    
    plt.plot(AirGo_breath.index[:1000], AirGo_breath.Band[:1000])
    plt.plot(resampled.DateTime[:10000], resampled.Band[:10000])
    plt.show()

In [8]:
filepath = 'C:/Users/wg984/Wolfgang/repos/ICU-Sleep/data/sample_airgo_rawv4.csv'
savepath = 'C:/Users/wg984/Wolfgang/repos/ICU-Sleep/data/sample_research_rawv4.csv'
filename = '003_ 2018_06_18pm_airgo raw AirGo 78a5043e9b82 2018-06-18 203845_Raw.csv'

In [24]:
# filename = '001_2018_06_06pm_airgo breath AirGo 78a5043e9810 2018-06-06 152600_Breath.csv'
AirGo_data = pd.read_csv(filepath)

In [25]:
AirGo_data.head()

,logId,sampleNumber,rawBeltValue,accX,accY,accZ
0,1,1,10352,250,254,17
1,1,2,10352,250,254,17
2,1,3,10328,250,255,17
3,1,4,10336,251,254,17
4,1,5,10352,250,255,16


In [26]:
AirGo_data.columns = AirGo_data.columns.str.strip()

starting_date = re.search("\d\d\d\d-\d\d-\d\d \d\d\d\d\d\d", filename)
starting_date = starting_date.group(0)
starting_date = datetime.datetime.strptime(starting_date, '%Y-%m-%d %H%M%S')
starting_date

datetime.datetime(2018, 6, 18, 20, 38, 45)

In [27]:
AirGo_data['DateTime'] = [starting_date + datetime.timedelta(seconds= x/10) for x in AirGo_data.sampleNumber.values]

In [29]:
AirGo_data.head()

,logId,sampleNumber,rawBeltValue,accX,accY,accZ,DateTime
0,1,1,10352,250,254,17,2018-06-18 20:38:45.100
1,1,2,10352,250,254,17,2018-06-18 20:38:45.200
2,1,3,10328,250,255,17,2018-06-18 20:38:45.300
3,1,4,10336,251,254,17,2018-06-18 20:38:45.400
4,1,5,10352,250,255,16,2018-06-18 20:38:45.500


In [32]:
AirGo_data = AirGo_data[['DateTime','rawBeltValue','accX','accY','accZ']]
AirGo_data.columns = ['DateTime','Band','accX','accY','accZ']

In [33]:
AirGo_data.head()

,DateTime,Band,accX,accY,accZ
0,2018-06-18 20:38:45.100,10352,250,254,17
1,2018-06-18 20:38:45.200,10352,250,254,17
2,2018-06-18 20:38:45.300,10328,250,255,17
3,2018-06-18 20:38:45.400,10336,251,254,17
4,2018-06-18 20:38:45.500,10352,250,255,16


In [ ]:
# drop duplicate:
dup = resampled.duplicated(subset=['DateTime','Band'])
resampled = resampled.mask(dup).dropna(axis = 0, how='all')

# detect all open belt areas
OpenBeltIdx = np.where(resampled['Band'] > 60000)[0]

LastIdxOpenBeltAtBeginOfRecording = np.where(np.diff(OpenBeltIdx)>1)[0][0]
LastIdxOpenBeltAtBeginOfRecording = OpenBeltIdx[LastIdxOpenBeltAtBeginOfRecording]
FirstIdxOpenBeltAtEndOfRecording = np.where(np.diff(OpenBeltIdx)>1)[0][-1]
FirstIdxOpenBeltAtEndOfRecording  = OpenBeltIdx[FirstIdxOpenBeltAtEndOfRecording+1]
resampled = resampled.iloc[LastIdxOpenBeltAtBeginOfRecording+1:FirstIdxOpenBeltAtEndOfRecording,:]

resampled['CRCStatus'] = np.nan

resampled.to_csv(savepath, index = False)

plt.plot(AirGo_breath.index[:1000], AirGo_breath.Band[:1000])
plt.plot(resampled.DateTime[:10000], resampled.Band[:10000])
plt.show()